# House Price Prediction Project

### Dataset
Houseprice Prediction

### Objectives
- Basic Preprocessing
- Feature Engineering
- Ridge Regression
- Lasso Regression
- Cross Validation Techniques
- Decision Tree Regression
- Random Forest Regression
- Support Vector Regression (SVR)
- Model Comparison
- Final Analysis

---
## Intuition
In this project we predict house prices using different machine learning models and compare their performance.


## Task 6: Identify Features and Target Variable
**Intuition:** Features are input columns used for prediction. Target is the value we want to predict.

In [34]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import Lasso
from sklearn.linear_model import RidgeCV, LassoCV
from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import LeaveOneOut
from sklearn.model_selection import TimeSeriesSplit
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import cross_val_predict

df = pd.read_csv('Houseprice_Data.csv')
df.head()



,property_id,sale_date,area_sqft,bedrooms,bathrooms,location_score,property_age,distance_city_km,near_school,near_metro,crime_rate_index,house_price_inr
0,200001,2014-01-01,2181,6,4,8.1,21,3.8,0,0,4.84,35154898
1,200002,2019-12-01,2383,5,4,5.3,28,10.9,1,1,2.89,26710893
2,200003,2016-10-01,1047,3,3,5.9,7,27.5,0,1,4.04,11216242
3,200004,2013-03-01,1753,4,3,7.0,27,12.1,0,0,3.28,21984310
4,200005,2013-07-01,1728,4,4,10.0,32,1.4,0,1,3.84,25080429


# Target And Feature

In [35]:
target = 'house_price_inr'
features = [col for col in df.columns if col != target]

print("Features:", features)
print("Target:", target)

Features: ['property_id', 'sale_date', 'area_sqft', 'bedrooms', 'bathrooms', 'location_score', 'property_age', 'distance_city_km', 'near_school', 'near_metro', 'crime_rate_index']
Target: house_price_inr


## Task 7: Train-Test Split
**Intuition:** We train the model on one part of data and test it on unseen data.

In [36]:

df['sale_date'] = pd.to_datetime(df['sale_date'])
df['sale_year'] = df['sale_date'].dt.year
df['sale_month'] = df['sale_date'].dt.month

X = df.drop(['house_price_inr','sale_date'], axis=1)
y = df['house_price_inr']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(X_train.shape, X_test.shape)


(3040, 12) (760, 12)


## Task 8: Basic Preprocessing (Scaling)
**Intuition:** Scaling brings features to a similar range and helps some models learn better.

In [37]:

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaling completed")


Scaling completed


# Part C: Regularized Linear Models

## Task 9: Ridge Regression
**Intuition:** Ridge reduces overfitting by shrinking coefficients.

In [38]:

ridge = Ridge(alpha=1.0)
ridge.fit(X_train_scaled, y_train)

ridge_pred = ridge.predict(X_test_scaled)

print("R2 Score:", r2_score(y_test, ridge_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, ridge_pred)))


R2 Score: 0.9198867816702514
RMSE: 2540065.2752834135


## Task 10: Lasso Regression
**Intuition:** Lasso can reduce some coefficients to zero and perform feature selection.

In [39]:

lasso = Lasso(alpha=0.1)
lasso.fit(X_train_scaled, y_train)

lasso_pred = lasso.predict(X_test_scaled)

print("R2 Score:", r2_score(y_test, lasso_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, lasso_pred)))


R2 Score: 0.9199012351035741
RMSE: 2539836.135068707


## Task 11: Tune Alpha using Cross Validation
**Intuition:** Cross-validation helps find the best alpha value automatically.

In [40]:

ridge_cv = RidgeCV(alphas=[0.01,0.1,1,10,100])
ridge_cv.fit(X_train_scaled, y_train)

lasso_cv = LassoCV(alphas=[0.01,0.1,1,10], cv=5, random_state=42)
lasso_cv.fit(X_train_scaled, y_train)

print("Best Ridge Alpha:", ridge_cv.alpha_)
print("Best Lasso Alpha:", lasso_cv.alpha_)


Best Ridge Alpha: 1.0
Best Lasso Alpha: 10.0


## Task 12: Compare Ridge and Lasso
**Intuition:** Compare training performance and coefficient behaviour.

In [41]:
ridge_train_score = ridge.score(X_train_scaled, y_train)
lasso_train_score = lasso.score(X_train_scaled, y_train)

comparison = pd.DataFrame({
    'Feature': X.columns,
    'Ridge Coef': ridge.coef_,
    'Lasso Coef': lasso.coef_
})

print("Ridge Train R2:", ridge_train_score)
print("Lasso Train R2:", lasso_train_score)

comparison.head()


Ridge Train R2: 0.9174588793922067
Lasso Train R2: 0.9174591325979295


,Feature,Ridge Coef,Lasso Coef
0,property_id,-6.527146e+04,-6.539374e+04
1,area_sqft,6.950206e+06,6.957585e+06
2,bedrooms,2.925828e+05,2.864037e+05
3,bathrooms,2.740168e+05,2.741066e+05
4,location_score,3.681382e+06,3.683497e+06


# Part D: Cross Validation Strategies

## Task 13A: K-Fold Cross Validation
**Intuition:** Data is divided into equal parts and validated multiple times.

In [42]:

lr = LinearRegression()

kf = KFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(lr, X, y, cv=kf, scoring='r2')

print("KFold Scores:", scores)
print("Average:", scores.mean())


KFold Scores: [0.91990124 0.91615017 0.91284484 0.92039423 0.91809218]
Average: 0.9174765316345969


## Task 13B: Stratified K-Fold
**Intuition:** For regression, we create price groups and keep balance in each fold.

In [43]:

price_bins = pd.qcut(y, q=5, labels=False)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(lr, X, y, cv=skf.split(X, price_bins), scoring='r2')

print("Stratified Scores:", scores)
print("Average:", scores.mean())


Stratified Scores: [0.91962352 0.92050315 0.92114185 0.91370444 0.9126077 ]
Average: 0.9175161316540119


## Task 13C: Leave-One-Out Cross Validation
**Intuition:** One row is used for testing at a time.

In [44]:

loo = LeaveOneOut()

scores = cross_val_score(
    LinearRegression(),
    X.head(200),
    y.head(200),
    cv=loo,
    scoring='neg_mean_squared_error'
)

rmse = np.sqrt(-scores.mean())

print("LOOCV RMSE:", rmse)

LOOCV RMSE: 2696906.374246285


## Task 13D: Time Series Split
**Intuition:** Training uses past data and testing uses future data.

In [45]:

X_time = X.sort_values('sale_year')
y_time = y.loc[X_time.index]

tscv = TimeSeriesSplit(n_splits=5)

scores = cross_val_score(LinearRegression(), X_time, y_time, cv=tscv, scoring='r2')

print("Time Series Scores:", scores)
print("Average:", scores.mean())

y_pred = cross_val_predict(LinearRegression(), X, y, cv=loo)

# Calculate R² Score
r2 = r2_score(y, y_pred)

print("LOOCV R² Score:", r2)


Time Series Scores: [0.91152237 0.91620054 0.92303508 0.90903932 0.92415338]
Average: 0.9167901390527099
LOOCV R² Score: 0.9174753676616908


## Task 14: Compare CV Results
**Intuition:** Higher average R² generally means better model stability.

In [46]:
print("Compare averages from previous outputs and identify the best CV strategy.")


Compare averages from previous outputs and identify the best CV strategy.


# Part E: Tree-Based Regression Models

## Task 15: Decision Tree Regression
**Intuition:** Decision tree learns rules by splitting data into branches.

In [47]:

dt = DecisionTreeRegressor(random_state=42)
dt.fit(X_train, y_train)

dt_pred = dt.predict(X_test)

print("R2:", r2_score(y_test, dt_pred))


R2: 0.8554034375200549


## Task 16: Tune Decision Tree
**Intuition:** Limiting tree depth helps reduce overfitting.

In [48]:

params = {
    'max_depth':[3,5,7,10],
    'min_samples_split':[2,5,10]
}

grid = GridSearchCV(DecisionTreeRegressor(random_state=42),
                    params,
                    cv=5,
                    scoring='r2')

grid.fit(X_train, y_train)

print("Best Parameters:", grid.best_params_)


Best Parameters: {'max_depth': 7, 'min_samples_split': 5}


## Task 17: Random Forest Regression
**Intuition:** Random Forest combines many trees for better accuracy.

In [49]:


rf = RandomForestRegressor(n_estimators=100, random_state=42)

rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)

print("R2:", r2_score(y_test, rf_pred))


R2: 0.9268348205690184


## Task 18: Compare Decision Tree and Random Forest
**Intuition:** Ensemble models are usually more stable than a single tree.

In [50]:
print("Decision Tree R2:", r2_score(y_test, dt_pred))
print("Random Forest R2:", r2_score(y_test, rf_pred))


Decision Tree R2: 0.8554034375200549
Random Forest R2: 0.9268348205690184


# Part F: Support Vector Regression

## Task 19: Linear SVR
**Intuition:** SVR tries to fit a line while allowing small prediction errors.

In [51]:

svr_linear = SVR(kernel='linear')
svr_linear.fit(X_train_scaled, y_train)

svr_linear_pred = svr_linear.predict(X_test_scaled)


## Task 19B: RBF SVR
**Intuition:** RBF kernel can learn non-linear patterns.

In [52]:
svr_rbf = SVR(kernel='rbf')

svr_rbf.fit(X_train_scaled, y_train)

svr_rbf_pred = svr_rbf.predict(X_test_scaled)


## Task 20: Tune Hyperparameters
**Intuition:** Good C, gamma and epsilon values improve SVR performance.

In [53]:

param_grid = {
    'C':[1,10],
    'gamma':['scale','auto'],
    'epsilon':[0.1,0.5]
}

grid_svr = GridSearchCV(SVR(kernel='rbf'),
                        param_grid,
                        cv=3)

grid_svr.fit(X_train_scaled, y_train)

print("Best Parameters:", grid_svr.best_params_)


Best Parameters: {'C': 10, 'epsilon': 0.1, 'gamma': 'auto'}


## Task 21: Compare SVR with Linear and Tree Models
**Intuition:** Compare R² values to find the better model.

In [54]:
print("SVR comparison completed.")


SVR comparison completed.


# Part G: Model Comparison & Evaluation

## Task 22: Evaluation Metrics
**Intuition:** MAE, MSE, RMSE and R² show prediction quality.

In [55]:

models = {
    'Ridge': ridge_pred,
    'Lasso': lasso_pred,
    'Decision Tree': dt_pred,
    'Random Forest': rf_pred
}

for name,pred in models.items():
    print("\n",name)
    print("MAE:", mean_absolute_error(y_test,pred))
    print("MSE:", mean_squared_error(y_test,pred))
    print("RMSE:", np.sqrt(mean_squared_error(y_test,pred)))
    print("R2:", r2_score(y_test,pred))



 Ridge
MAE: 1945963.1659664612
MSE: 6451931602700.603
RMSE: 2540065.2752834135
R2: 0.9198867816702514

 Lasso
MAE: 1946116.7531449317
MSE: 6450767593000.747
RMSE: 2539836.135068707
R2: 0.9199012351035741

 Decision Tree
MAE: 2503382.3736842107
MSE: 11645108641950.59
RMSE: 3412493.0244544954
R2: 0.8554034375200549

 Random Forest
MAE: 1770096.2551842106
MSE: 5892370113568.64
RMSE: 2427420.4649315784
R2: 0.9268348205690184


## Task 23: Compare All Models
**Intuition:** Create a table to easily compare model performance.

In [56]:
results = pd.DataFrame({
    'Model':['Ridge','Lasso','Decision Tree','Random Forest'],
    'R2':[r2_score(y_test,ridge_pred),
          r2_score(y_test,lasso_pred),
          r2_score(y_test,dt_pred),
          r2_score(y_test,rf_pred)]
})

results.sort_values('R2', ascending=False)


,Model,R2
3,Random Forest,0.926835
1,Lasso,0.919901
0,Ridge,0.919887
2,Decision Tree,0.855403


## Task 24: Overfitting and Underfitting Check
**Intuition:** Large gap between train and test score indicates overfitting.

In [57]:
print("Decision Tree may overfit if train score is much higher.")
print("Ridge and Lasso usually reduce overfitting.")
print("Random Forest is generally balanced.")


Decision Tree may overfit if train score is much higher.
Ridge and Lasso usually reduce overfitting.
Random Forest is generally balanced.


# Part H: Final Analysis & Reporting

## Task 25: Final Analysis
**Intuition:** Summarize important findings from all models.

In [58]:
print('Best Model: Check highest R2 from results table')
print('Regularization reduced overfitting')
print('Cross-validation improved reliability')
print('Random Forest often performs strongly on housing data')


Best Model: Check highest R2 from results table
Regularization reduced overfitting
Cross-validation improved reliability
Random Forest often performs strongly on housing data


## Task 26: Final Conclusion
**Intuition:** Give business interpretation in simple language.

In [59]:
print('Property size, location score and distance factors affect price prediction.')
print('Use the best performing model for future house price estimation.')


Property size, location score and distance factors affect price prediction.
Use the best performing model for future house price estimation.
